##  Load Dataset

In [7]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(r"F:\StressGNN")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "extended_stress_detection_data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_DATA_PATH)

TARGET = "Stress_Detection"

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target: {TARGET}")

Dataset: 3,000 rows × 22 columns
Target: Stress_Detection


## Validate dataset before splitting

In [8]:
assert TARGET in df.columns
assert df[TARGET].isna().sum() == 0
assert df.duplicated().sum() == 0

print("Dataset validation passed.")
print(f"Missing target values : {df[TARGET].isna().sum()}")
print(f"Duplicate rows        : {df.duplicated().sum()}")
print("\nTarget distribution:")

display(
    df[TARGET]
    .value_counts()
    .rename_axis("Stress Level")
    .to_frame("Count")
)

Dataset validation passed.
Missing target values : 0
Duplicate rows        : 0

Target distribution:


,Count
Stress Level,
Medium,1258
High,1118
Low,624


## Stratified Split

In [9]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

train_dev, test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[TARGET]
)

train_dev = train_dev.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"Full dataset : {len(df):,}")
print(f"Train/Dev    : {len(train_dev):,}")
print(f"Test         : {len(test):,}")

Full dataset : 3,000
Train/Dev    : 2,400
Test         : 600


## Verify split integrity

In [10]:
distribution = pd.DataFrame({
    "Full Dataset (%)": df[TARGET].value_counts(normalize=True),
    "Train/Dev (%)": train_dev[TARGET].value_counts(normalize=True),
    "Test (%)": test[TARGET].value_counts(normalize=True)
}).sort_index() * 100

display(distribution.round(2))

# Verify total row count
assert len(train_dev) + len(test) == len(df)

# Verify expected split sizes
assert len(train_dev) == 2400
assert len(test) == 600

# Verify no duplicated rows between splits
train_hashes = pd.util.hash_pandas_object(train_dev, index=False)
test_hashes = pd.util.hash_pandas_object(test, index=False)

assert len(set(train_hashes) & set(test_hashes)) == 0

print("✓ Row-count verification passed.")
print("✓ Expected 80/20 split confirmed.")
print("✓ No train/test row overlap detected.")
print("✓ Stratified class distribution preserved.")

,Full Dataset (%),Train/Dev (%),Test (%)
Stress_Detection,,,
High,37.27,37.25,37.33
Low,20.80,20.79,20.83
Medium,41.93,41.96,41.83


✓ Row-count verification passed.
✓ Expected 80/20 split confirmed.
✓ No train/test row overlap detected.
✓ Stratified class distribution preserved.


In [11]:
TRAIN_DEV_PATH = PROCESSED_DIR / "train_dev.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
SPLIT_INFO_PATH = PROCESSED_DIR / "split_info.json"

train_dev.to_csv(TRAIN_DEV_PATH, index=False)
test.to_csv(TEST_PATH, index=False)

split_info = {
    "source_dataset": RAW_DATA_PATH.name,
    "total_samples": len(df),
    "train_dev_samples": len(train_dev),
    "test_samples": len(test),
    "test_ratio": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "target": TARGET,
    "stratified": True,
    "missing_values_in_raw": int(df.isna().sum().sum()),
    "duplicate_rows_in_raw": int(df.duplicated().sum())
}

with open(SPLIT_INFO_PATH, "w") as f:
    json.dump(split_info, f, indent=4)

print("DATA SPLIT COMPLETE")
print("-" * 40)
print(f"Train/Dev : {TRAIN_DEV_PATH}")
print(f"Test      : {TEST_PATH}")
print(f"Metadata  : {SPLIT_INFO_PATH}")
print()
print("Test set is now FROZEN for final evaluation.")

DATA SPLIT COMPLETE
----------------------------------------
Train/Dev : F:\StressGNN\data\processed\train_dev.csv
Test      : F:\StressGNN\data\processed\test.csv
Metadata  : F:\StressGNN\data\processed\split_info.json

Test set is now FROZEN for final evaluation.
